# Round 2: the three experiments the paper still needs (Colab T4)

| Part | What it answers | Runs | Time |
|---|---|---|---|
| A. Dose-response | Does M2-CL change anything once the contrastive loss is strong? | 4 (Art, seed 0, alpha 0.01, 0.1, 0.3, 1.0) | about 2 h |
| B. Learning rate | Does the paper's lr 1e-3 change M2's accuracy? | 4 (M2, seed 0, all folds) | about 1.8 h |
| C. ERM, corrected split | A fair ERM baseline for the corrected M2 and M2-CL runs | 12 (all folds, seeds 0-2) | about 3-5 h |

Run them in this order: A gives the most information per hour. Each part saves to its own folder in `My Drive/m2cl_v2/` (`dose/`, `lr/`, `erm/`), so finished runs are never overwritten, and every run resumes from its last saved epoch.

**Before you start**, upload to the top level of **My Drive**:
1. `M2CL_code.zip`, from the repo's `colab_upload/` folder. The setup cell stops if you use the old copy in `Downloads`.
2. `PACS.zip`

Then choose Runtime > Change runtime type > **T4 GPU** and run the cells in order.

**If Colab stops or this account runs out:** download the `m2cl_v2` folder from this account's Drive, upload the zip to the next account's My Drive together with the two files above (don't rename it), and run all the cells again there. The restore cell picks up every finished and half-finished run.

**Don't choose settings from the test accuracy these runs print.** The paper selects by source-domain validation accuracy only.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

DRIVE    = '/content/drive/MyDrive'
CODE_ZIP = f'{DRIVE}/M2CL_code.zip'
PACS_ZIP = f'{DRIVE}/PACS.zip'
OUT      = f'{DRIVE}/m2cl_v2'

import os, shutil, zipfile
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
for path in (CODE_ZIP, PACS_ZIP):
    assert os.path.exists(path), f'Not found: {path}. Upload it to the top level of My Drive.'

shutil.rmtree('/content/repo', ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/repo')
os.chdir('/content/repo')
assert 'latest_prev' in open('src/m2cl/engine.py', encoding='utf-8').read(), \
    'This is the old M2CL_code.zip. Upload the one from the colab_upload folder and run this cell again.'
!pip -q install pyyaml tabulate

if not os.path.exists('/content/PACS_raw'):
    print('Extracting PACS...')
    zipfile.ZipFile(PACS_ZIP).extractall('/content/PACS_raw')

# Use the real images: the folder of four domains whose files are not dct_* (docs/AUDIT.md, A0)
def is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
matches = [r for r, d, _ in os.walk('/content/PACS_raw')
           if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and is_rgb(r)]
assert matches, 'No RGB copy of PACS found in PACS.zip.'
DATA = matches[0]
n_images = sum(len(f) for _, _, f in os.walk(DATA))
print('PACS root:', DATA, '| images:', n_images)
assert n_images == 9991, f'Expected 9,991 images, found {n_images}.'
os.makedirs(OUT, exist_ok=True)

free_gb = shutil.disk_usage(DRIVE).free / 1e9
print(f'Free Drive space: {free_gb:.1f} GB')
if free_gb < 5:
    print('WARNING: less than 5 GB free. Empty the Drive trash before training.')

In [ ]:
# 2. Only after switching accounts: copy round-2 progress out of an uploaded m2cl_v2 zip.
# Nothing to do on the first account. Files already here are kept, so rerunning is safe.
import glob

PARTS = ('dose', 'lr', 'erm')
found = sorted(glob.glob(f'{DRIVE}/m2cl_v2*.zip'), key=os.path.getmtime, reverse=True)
copied = 0
for path in found:
    with zipfile.ZipFile(path) as z:
        for info in z.infolist():
            parts = info.filename.replace('\\', '/').split('/')
            if info.filename.endswith('/') or parts[-1].endswith('.tmp'):
                continue
            starts = [i for i, p in enumerate(parts[:-1]) if p in PARTS]
            if not starts:
                continue
            target = os.path.join(OUT, *parts[starts[0]:])
            if os.path.exists(target):
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target + '.tmp', 'wb') as dst:
                shutil.copyfileobj(src, dst)
            os.replace(target + '.tmp', target)
            copied += 1
print(f'Read {len(found)} zip(s); copied {copied} files.')

In [ ]:
# 3. Where each run stands. Run this any time.
import json
import pandas as pd

TARGETS = ('art_painting', 'cartoon', 'photo', 'sketch')
ALPHAS = (0.01, 0.1, 0.3, 1.0)
PLAN = ([('A. dose', f'alpha {a}', f'{OUT}/dose/alpha_{a}/m2cl_6tap_art_painting_seed0') for a in ALPHAS]
        + [('B. lr 1e-3', t, f'{OUT}/lr/lr_0.001/m2_6tap_{t}_seed0') for t in TARGETS]
        + [('C. ERM', f'{t} seed {s}', f'{OUT}/erm/erm_{t}_seed{s}') for t in TARGETS for s in (0, 1, 2)])

def run_status(run_dir):
    result = os.path.join(run_dir, 'result.json')
    if os.path.exists(result) and json.load(open(result)).get('status') == 'complete':
        return 'finished'
    for name in ('latest.pt', 'latest_prev.pt'):
        try:
            return f"continues at epoch {torch.load(os.path.join(run_dir, name), map_location='cpu', weights_only=False)['epoch'] + 1}"
        except Exception:
            pass
    return 'not started'

status = pd.DataFrame([{'part': p, 'run': r, 'status': run_status(d)} for p, r, d in PLAN])
display(status)
print((status.status == 'finished').sum(), 'of', len(status), 'runs finished')

## A. Dose-response: M2-CL with a stronger contrastive loss
Same settings as the locked M2-CL (6 taps, tau 1.0, lr 5e-4, corrected split, Art held out, seed 0), with alpha raised from 0.003. At alpha 0.003 the contrastive gradient was about 0.4% of the cross-entropy gradient; these values should take it to roughly 1.5%, 13%, 40% and 130%. Each run takes about 30 minutes. Watch the `cl=` value: at alpha 0.003 it never left about 170.

In [ ]:
for a in ALPHAS:
    !python scripts/train_lodo.py --config configs/m2cl_6tap.yaml --alpha {a} --targets art_painting --seeds 0 --data_root "$DATA" --out_dir "$OUT/dose/alpha_{a}"

## B. Learning rate 1e-3
6-tap M2 exactly as before, but with the paper's learning rate of 1e-3 instead of 5e-4 (cosine schedule kept), seed 0 on all four folds. Compare with the existing M2 seed-0 runs at 5e-4.

In [ ]:
!python scripts/train_lodo.py --config configs/m2_6tap.yaml --lr 0.001 --seeds 0 --data_root "$DATA" --out_dir "$OUT/lr/lr_0.001"

## C. ERM with the corrected validation split
Plain ResNet-18, same lr, batch, epochs and split as the corrected M2 and M2-CL runs, three seeds on all four folds.

In [ ]:
!python scripts/train_lodo.py --config configs/erm.yaml --data_root "$DATA" --out_dir "$OUT/erm"

In [ ]:
# 4. Results so far
def load(run_dir):
    path = os.path.join(run_dir, 'result.json')
    if not os.path.exists(path):
        return None
    r = json.load(open(path))
    return r if r.get('status') == 'complete' else None

# Existing corrected-split runs, for comparison (docs/AUDIT.md, F9)
M2_SEED0 = {'art_painting': 73.39, 'cartoon': 68.47, 'photo': 95.33, 'sketch': 61.92}
M2_ART0_VAL, M2CL_ART0 = 94.64, {'alpha': 0.003, 'test': 73.24, 'best_val': 94.68}

rows = [{'alpha': 0.0, 'best_val': M2_ART0_VAL, 'test': M2_SEED0['art_painting'], 'note': 'existing M2'},
        {'alpha': 0.003, 'best_val': M2CL_ART0['best_val'], 'test': M2CL_ART0['test'], 'note': 'existing M2-CL'}]
for a in ALPHAS:
    r = load(f'{OUT}/dose/alpha_{a}/m2cl_6tap_art_painting_seed0')
    if r:
        h = r['history']
        rows.append({'alpha': a, 'best_val': round(r['best_val'], 2), 'test': round(r['test_acc'], 2),
                     'best_epoch': r['best_epoch'], 'cl first': round(h[0]['cl_loss'], 1),
                     'cl last': round(h[-1]['cl_loss'], 1), 'note': 'new'})
print('A. Dose-response, Art held out, seed 0')
display(pd.DataFrame(rows))

rows = []
for t in TARGETS:
    r = load(f'{OUT}/lr/lr_0.001/m2_6tap_{t}_seed0')
    if r:
        rows.append({'target': t, 'lr 5e-4': M2_SEED0[t], 'lr 1e-3': round(r['test_acc'], 2),
                     'difference': round(r['test_acc'] - M2_SEED0[t], 2), 'best_epoch': r['best_epoch']})
print('B. M2 seed 0, learning rate')
display(pd.DataFrame(rows))

rows = []
for t in TARGETS:
    for s in (0, 1, 2):
        r = load(f'{OUT}/erm/erm_{t}_seed{s}')
        if r:
            c = r['config']
            assert (c['model'], c['lr'], c['batch_size'], c['epochs']) == ('erm', 0.0005, 128, 30), f'Wrong settings for {t} seed {s}'
            rows.append({'target': t, 'seed': s, 'test_acc': r['test_acc'], 'torch': r['torch']})
erm = pd.DataFrame(rows)
print(f'C. ERM, corrected split: {len(erm)} of 12 runs complete')
if len(erm):
    display(erm.pivot(index='target', columns='seed', values='test_acc').round(2))
    print(erm.groupby('target').test_acc.agg(['mean', 'std']).round(2))
    if len(erm) == 12:
        print('Macro-average:', round(erm.groupby('target').test_acc.mean().mean(), 2))

## When you're done
Download `My Drive/m2cl_v2/dose`, `m2cl_v2/lr` and `m2cl_v2/erm` (or the whole `m2cl_v2` folder) to the laptop and tell Claude which runs, if any, printed a `[resume]` line. Claude then measures each dose-response checkpoint's gradient share and its distance from the paired M2 run, and fills the three `[PENDING]` markers in `paper/M2CL_v3.tex`.